# 2. SQLite数据库建立与数据写入

## 2.1 任务说明
本notebook完成以下任务：
- 创建SQLite数据库文件`fin_data.db`
- 按照指定表结构建立宏数据表、股票行情表、股票信息表
- 将01notebook获取的数据写入数据库
- 编写增量更新函数，避免重复下载

## 2.2 数据库连接与表创建

### 前置说明
使用sqlite3连接数据库，按任务要求的表结构创建三张表：macro_data（宏观数据）、stock_price（股票行情）、stock_info（股票信息）。

In [ ]:
import sqlite3
import pandas as pd
import os

# 数据库文件路径
db_path = "fin_data.db"

# 连接数据库（不存在则自动创建）
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 1. 创建宏观数据表
cursor.execute("""
    CREATE TABLE IF NOT EXISTS macro_data (
        date        TEXT NOT NULL,
        series_id   TEXT NOT NULL,
        value       REAL,
        PRIMARY KEY (date, series_id)
    )
""")

# 2. 创建股票行情表
cursor.execute("""
    CREATE TABLE IF NOT EXISTS stock_price (
        code        TEXT NOT NULL,
        date        TEXT NOT NULL,
        open        REAL,
        high        REAL,
        low         REAL,
        close       REAL,
        volume      REAL,
        amount      REAL,
        adjustflag  TEXT,
        PRIMARY KEY (code, date)
    )
""")

# 3. 创建股票信息表
cursor.execute("""
    CREATE TABLE IF NOT EXISTS stock_info (
        code        TEXT PRIMARY KEY,
        code_name   TEXT,
        ipo_date    TEXT,
        industry    TEXT,
        market      TEXT
    )
""")

conn.commit()
print("数据库表创建完成")

### 结果解读
- 三个表已创建，带PRIMARY KEY约束防止重复插入
- macro_data: date + series_id 联合主键
- stock_price: code + date 联合主键
- stock_info: code 为主键

## 2.3 写入宏观数据

### 前置说明
将FRED获取的宏观数据（宽格式）转换为长格式（date, series_id, value），写入macro_data表。

In [ ]:
# 读取之前保存的宏观数据
macro_df = pd.read_csv("output/macro_data.csv", index_col=0, parse_dates=True)

# 宽格式转为长格式（melt操作）
macro_long = macro_df.reset_index().melt(
    id_vars='date',
    var_name='series_id',
    value_name='value'
)
macro_long['date'] = macro_long['date'].dt.strftime('%Y-%m-%d')

print(f"宏观数据长格式形状: {macro_long.shape}")
print(macro_long.head())

# 写入数据库（IGNORE处理重复主键）
macro_long.to_sql('macro_data', conn, if_exists='replace', index=False)

# 验证写入
result = pd.read_sql_query("SELECT COUNT(*) FROM macro_data", conn)
print(f"\n写入macro_data: {result.iloc[0,0]} 条记录")

### 结果解读
- 宽格式(314, 6)转为长格式(1884, 3)
- 每个日期对应6个序列（series_id）
- if_exists='replace'会覆盖旧数据，生产环境建议用'append'

## 2.4 写入股票行情数据

### 前置说明
将沪深300指数和自选股票行情数据写入stock_price表。

In [ ]:
# 读取股票行情数据
hs300_df = pd.read_csv("output/hs300_daily.csv")
stock_df = pd.read_csv("output/stock_daily.csv")

# 合并沪深300和自选股
all_stock_price = pd.concat([hs300_df, stock_df], ignore_index=True)

# 写入数据库
all_stock_price.to_sql('stock_price', conn, if_exists='replace', index=False)

# 验证写入
result = pd.read_sql_query("SELECT COUNT(*) FROM stock_price", conn)
print(f"写入stock_price: {result.iloc[0,0]} 条记录")

# 按股票统计
stock_count = pd.read_sql_query("""
    SELECT code, COUNT(*) as cnt 
    FROM stock_price 
    GROUP BY code
    ORDER BY cnt DESC
""", conn)
print("\n各股票记录数:")
print(stock_count)

### 结果解读
- 所有股票数据已写入stock_price表
- 记录总数应等于沪深300 + 各自选股记录之和
- 可查看各股票的数据量是否合理（约250交易日/年）

## 2.5 写入股票基本信息

### 前置说明
将股票基本信息写入stock_info表。

In [ ]:
# 读取股票信息
stock_info_df = pd.read_csv("output/stock_info.csv")

print("股票基本信息:")
print(stock_info_df)

# 写入数据库
stock_info_df.to_sql('stock_info', conn, if_exists='replace', index=False)

# 验证写入
result = pd.read_sql_query("SELECT COUNT(*) FROM stock_info", conn)
print(f"\n写入stock_info: {result.iloc[0,0]} 条记录")

### 结果解读
- 包含每只股票的代码、名称、上市日期、行业、市场等信息
- 上市日期可用于计算股票上市时长

## 2.6 数据库表结构验证

### 前置说明
查看数据库中各表的结构、数据量，确认写入正确。

In [ ]:
print("=" * 60)
print("数据库表结构验证")
print("=" * 60)

for table in ['macro_data', 'stock_price', 'stock_info']:
    # 获取表结构
    schema = pd.read_sql_query(f"PRAGMA table_info({table})", conn)
    print(f"\n【{table}】")
    print(schema[['name', 'type', 'notnull', 'pk']])
    
    # 获取记录数
    count = pd.read_sql_query(f"SELECT COUNT(*) FROM {table}", conn)
    print(f"记录数: {count.iloc[0,0]}")

### 结果解读
- macro_data: 1884条宏观数据记录
- stock_price: 所有股票行情记录总和
- stock_info: 股票基本信息记录

## 2.7 增量更新函数

### 前置说明
编写数据更新函数，检测数据库中各表的最新日期，只下载增量数据而非全量下载，提高效率。

In [ ]:
def get_latest_date(conn, table, date_column):
    """
    获取数据库表中指定日期列的最新日期
    
    参数:
        conn: 数据库连接
        table: 表名
        date_column: 日期列名
    返回:
        最新日期字符串，或None（表为空）
    """
    query = f"SELECT MAX({date_column}) FROM {table}"
    result = pd.read_sql_query(query, conn)
    latest = result.iloc[0, 0]
    return latest

def update_macro_data(conn, new_data):
    """
    增量更新宏观数据
    
    参数:
        conn: 数据库连接
        new_data: 新增的DataFrame数据
    """
    # 宽格式转长格式
    new_long = new_data.reset_index().melt(
        id_vars='date',
        var_name='series_id',
        value_name='value'
    )
    new_long['date'] = pd.to_datetime(new_long['date']).dt.strftime('%Y-%m-%d')
    
    # 追加写入（PRIMARY KEY自动过滤重复）
    new_long.to_sql('macro_data', conn, if_exists='append', index=False)
    
    print(f"新增 {len(new_long)} 条宏观数据记录")

# 示例：检查最新日期
latest_macro = get_latest_date(conn, 'macro_data', 'date')
print(f"宏观数据最新日期: {latest_macro}")

### 结果解读
- `get_latest_date`: 查询表中最新日期
- `update_macro_data`: 将新数据增量写入（重复主键自动跳过）
- 使用PRIMARY KEY约束确保不插入重复记录

## 2.8 数据库查询示例

### 前置说明
简单演示如何从数据库查询数据，验证数据可用性。

In [ ]:
# 示例1: 查询最近3个月的宏观数据
recent_macro = pd.read_sql_query("""
    SELECT * FROM macro_data 
    WHERE date >= '2025-10-01'
    ORDER BY date, series_id
""", conn)

print("最近3个月宏观数据:")
print(recent_macro)

# 示例2: 查询特定股票的最近行情
recent_price = pd.read_sql_query("""
    SELECT * FROM stock_price 
    WHERE code = 'sh.600519' 
    ORDER BY date DESC 
    LIMIT 5
""", conn)

print("\n贵州茅台最近5条行情:")
print(recent_price)

### 结果解读
- SQL查询可灵活获取任意范围的数据
- pandas.read_sql_query直接返回DataFrame便于分析

In [ ]:
# 关闭数据库连接
conn.close()
print("数据库连接已关闭")

### 结果解读
数据库使用完毕，正常关闭连接。